<a href="https://colab.research.google.com/github/lcatu/cccd_extraction_ai/blob/Chi/VietOCR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Kết nối với Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print("Google Drive đã kết nối!")

In [ ]:
# 2. Thiết lập môi trường bằng Token Kaggle
import os
os.environ['KAGGLE_USERNAME'] = "vulamnguyen"
os.environ['KAGGLE_KEY'] = "KGAT_175d45b8e0c942f80d35bf7caabd5ce4"

print("Đã cấu hình API Token thành công!")

In [ ]:
# 3. Tải dataset từ Kaggle
print("📥 Đang tải dataset (7.64 GB)...")
!kaggle datasets download -d vulamnguyen/vietocr-dataset -p /content
print("Tải xong!")

In [ ]:
# 4. tải đường dẫn VietOCR-Paddle/meta/
import zipfile, os, shutil

ZIP_PATH   = '/content/vietocr-dataset.zip'
TEMP_DIR   = '/content/viet_ocr_temp'
SAVE_DIR   = '/content/drive/MyDrive/VietOCR_Data'
OUTPUT_ZIP = '/content/VietOCR_meta.zip'

os.makedirs(TEMP_DIR, exist_ok=True)
os.makedirs(SAVE_DIR, exist_ok=True)

print("Bước 1/3: Giải nén folder meta vào Colab...")
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    targets = [
        f for f in zf.infolist()
        if not f.is_dir()
        and 'VietOCR-Paddle/meta/rec/' in f.filename  # ← SỬA chỗ này
        and ('/train/' in f.filename or f.filename.endswith('.txt'))
    ]
    total = len(targets)
    print(f"   Tìm thấy {total:,} files của meta")
    for i, f in enumerate(targets, 1):
        zf.extract(f, TEMP_DIR)
        if i % 20000 == 0:
            print(f"   {i:,}/{total:,} files ({i*100//total}%)")
print("Bước 1 xong!")

print("\n⏳ Bước 2/3: Đóng gói folder meta...")
with zipfile.ZipFile(OUTPUT_ZIP, 'w', zipfile.ZIP_DEFLATED) as zout:
    folder_path = os.path.join(TEMP_DIR, 'VietOCR-Paddle', 'meta')
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            file_abs = os.path.join(root, file)
            # Bỏ prefix VietOCR-Paddle/ khi lưu vào zip
            arcname = os.path.relpath(file_abs, os.path.join(TEMP_DIR, 'VietOCR-Paddle'))
            zout.write(file_abs, arcname)
zip_size = os.path.getsize(OUTPUT_ZIP) / (1024**3)
print(f"Bước 2 xong! Kích thước: {zip_size:.2f} GB")

print("\nBước 3/3: Đẩy lên Google Drive...")
shutil.copy2(OUTPUT_ZIP, f'{SAVE_DIR}/VietOCR_meta.zip')
print("XONG! Data đã an toàn trên Drive!")

shutil.rmtree(TEMP_DIR)
os.remove(ZIP_PATH)
os.remove(OUTPUT_ZIP)
print(" Đã dọn sạch bộ nhớ Colab!")

In [ ]:
# 5. (Dùng mỗi khi train): Giải nén từ Drive xuống Colab
import zipfile, os

ZIP_ON_DRIVE = '/content/drive/MyDrive/VietOCR_Data/VietOCR_meta.zip'
WORK_DIR     = '/content/viet_ocr_data'

os.makedirs(WORK_DIR, exist_ok=True)

print("Đang giải nén xuống Colab để train...")
with zipfile.ZipFile(ZIP_ON_DRIVE, 'r') as z:
    total = len(z.infolist())
    for i, f in enumerate(z.infolist(), 1):
        z.extract(f, WORK_DIR)
        if i % 20000 == 0:
            print(f"   {i:,}/{total:,} ({i*100//total}%)")

print(f"Sẵn sàng train! Data tại: {WORK_DIR}")
print(f"   Ảnh: {WORK_DIR}/meta/rec/train/")
print(f"   Nhãn: {WORK_DIR}/meta/rec/rec_gt_train.txt")

In [ ]:
# 6. kiểm tra GPU
import torch
print('=== GPU INFO ===')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Name: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('Không có GPU! Vào Runtime → Change runtime type → GPU')

In [ ]:
# 7. Cài thư viện và restart
!pip install scipy -q 2>&1 | tail -3
print("scipy xong")

!pip install Pillow -q 2>&1 | tail -3
print("Pillow xong")

!pip install editdistance scikit-learn tqdm -q 2>&1 | tail -3
print("utils xong")

!pip install vietocr -q 2>&1 | tail -3
print("vietocr xong")

!pip install imgaug -q 2>&1 | tail -3
print("imgaug xong")

# Cài thủ công các dep của vietocr
!pip install torch torchvision -q 2>&1 | tail -3
!pip install einops -q 2>&1 | tail -3
!pip install albumentations -q 2>&1 | tail -3
print("Tất cả xong!")
print("✅ Cài xong! Đang restart...")
import os
os.kill(os.getpid(), 9)

In [ ]:
# 8. Chạy SAU KHI restart
#Patch numpy
import numpy as np
if not hasattr(np, 'sctypes'):
    np.sctypes = {
        'int': [np.int8, np.int16, np.int32, np.int64],
        'uint': [np.uint8, np.uint16, np.uint32, np.uint64],
        'float': [np.float16, np.float32, np.float64],
        'complex': [np.complex64, np.complex128],
        'others': [bool, object, bytes, str]
    }

# Xóa cache LMDB ngay lập tức
import shutil, os
for d in ['/tmp/train_OCRDataset', '/tmp/valid_OCRDataset', '/tmp/test_OCRDataset']:
    if os.path.exists(d):
        shutil.rmtree(d)
        print(f"Xóa: {d}")

# Import
from vietocr.tool.config import Cfg
from vietocr.model.trainer import Trainer
import torch

print("Sẵn sàng!")

In [ ]:
# 9. Cấu hình đường dẫn và tham số
import torch, os, random

# Folder chứa ảnh
FOLDER_1 = '/content/viet_ocr_data/meta/rec'  # ← bỏ /train ở cuối

# Nếu có 1 file annotation CHUNG (để None nếu không có)
ANNOTATION_FILE = None

#Nếu mỗi folder có file annotation RIÊNG
ANNOTATION_FOLDER_1 = '/content/viet_ocr_data/meta/rec/rec_gt_train.txt'

# Thư mục lưu output (checkpoint, log, model)
OUTPUT_DIR = '/content/drive/MyDrive/vietocr_output'

#  Tỉ lệ chia dữ liệu
TRAIN_RATIO = 0.8   # 80%
VAL_RATIO   = 0.1   # 10%
TEST_RATIO  = 0.1   # 10%

#  Cấu hình training
BATCH_SIZE  = 32
NUM_EPOCHS  = 10
LR          = 0.0001
PRINT_EVERY = 200
VALID_EVERY = 1
# Model
MODEL_NAME = 'vgg_transformer'  # hoặc 'vgg_seq2seq'

#
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'✅ Cấu hình xong!')
print(f'   OUTPUT_DIR : {OUTPUT_DIR}')
print(f'   Train/Val/Test : {TRAIN_RATIO}/{VAL_RATIO}/{TEST_RATIO}')
print(f'   Model: {MODEL_NAME} | Batch: {BATCH_SIZE} | Epochs: {NUM_EPOCHS}')

In [ ]:
# 10. Tạo annotation + lọc vocab + lưu file
from vietocr.tool.config import Cfg

# Lấy vocab
_cfg_tmp = Cfg.load_config_from_name(MODEL_NAME)
vocab_chars = set(_cfg_tmp['vocab'])
print(f"Vocab: {len(vocab_chars)} ký tự")

# Đọc file gốc
samples = []
with open(ANNOTATION_FOLDER_1, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line: continue
        parts = line.split('\t', 1)
        if len(parts) != 2: continue
        img_name, label = parts[0].strip(), parts[1].strip()
        img_path = os.path.join(FOLDER_1, img_name)
        if os.path.exists(img_path):
            samples.append((img_path, label))

print(f"Load được: {len(samples):,} mẫu")
assert len(samples) > 0, "❌ 0 mẫu — kiểm tra đường dẫn!"

# Lọc ký tự ngoài vocab
before = len(samples)
samples = [(p, l) for p, l in samples if all(c in vocab_chars for c in l)]
print(f"Sau lọc vocab: {len(samples):,} (bỏ {before - len(samples):,})")

# Chia train/val/test
random.seed(42)
random.shuffle(samples)
n       = len(samples)
n_train = int(n * TRAIN_RATIO)
n_val   = int(n * VAL_RATIO)
train_s = samples[:n_train]
val_s   = samples[n_train:n_train+n_val]
test_s  = samples[n_train+n_val:]

# Lưu
TRAIN_ANNO = os.path.join(OUTPUT_DIR, 'train_annotation.txt')
VAL_ANNO   = os.path.join(OUTPUT_DIR, 'val_annotation.txt')
TEST_ANNO  = os.path.join(OUTPUT_DIR, 'test_annotation.txt')

for split, path in [(train_s, TRAIN_ANNO), (val_s, VAL_ANNO), (test_s, TEST_ANNO)]:
    with open(path, 'w', encoding='utf-8') as f:
        for img, lbl in split:
            f.write(f'{img}\t{lbl}\n')
    print(f"  ✅ {os.path.basename(path)}: {len(split):,} dòng")


In [ ]:
# 11. Khởi tạo OCRDataset
from torch.utils.data import Dataset
from PIL import Image

class OCRDataset(Dataset):
    def __init__(self, annotation_path):
        self.samples = []
        with open(annotation_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                parts = line.split('\t', 1)
                if len(parts) == 2:
                    self.samples.append((parts[0].strip(), parts[1].strip()))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        try:
            img = Image.open(img_path).convert('RGB')
        except Exception as e:
            print(f'Lỗi đọc ảnh {img_path}: {e}')
            img = Image.new('RGB', (100, 32), color=(255, 255, 255))
        return img, label


ds_train = OCRDataset(TRAIN_ANNO)
ds_val   = OCRDataset(VAL_ANNO)
ds_test  = OCRDataset(TEST_ANNO)

print(f'Dataset khởi tạo xong!')
print(f'   Train: {len(ds_train):,} | Val: {len(ds_val):,} | Test: {len(ds_test):,}')

img, lbl = ds_train[0]
print(f'   Mẫu thử — Kích thước: {img.size} | Nhãn: "{lbl}"')

In [ ]:
# 12. Config model
from vietocr.tool.config import Cfg
from vietocr.model.trainer import Trainer

config = Cfg.load_config_from_name(MODEL_NAME)

config['dataset'] = {
    'name'             : 'OCRDataset',
    'data_root'        : '',
    'train_annotation' : TRAIN_ANNO,
    'valid_annotation' : VAL_ANNO,
    'image_height'     : 32,
    'image_min_width'  : 32,
    'image_max_width'  : 512,
}

config['dataloader'] = {
    'num_workers'      : 2,
    'pin_memory'       : True,
}

config['trainer'] = {
    'iters'       : NUM_EPOCHS * len(ds_train) // BATCH_SIZE,
    'valid_every' : VALID_EVERY * len(ds_train) // BATCH_SIZE,
    'print_every' : PRINT_EVERY,
    'batch_size'  : BATCH_SIZE,
    'checkpoint'  : os.path.join(OUTPUT_DIR, 'checkpoint.pth'),
    'export'      : os.path.join(OUTPUT_DIR, 'best_model.pth'),
    'metrics'     : 10000,
    'log'         : os.path.join(OUTPUT_DIR, 'train.log'),
}

config['device']    = 'cuda' if torch.cuda.is_available() else 'cpu'
config['pretrained'] = True

print('Cấu hình model xong!')
print(f'   Model     : {MODEL_NAME}')
print(f'   Device    : {config["device"]}')
print(f'   Tổng iter : {config["trainer"]["iters"]:,}')
print(f'   Checkpoint: {config["trainer"]["checkpoint"]}')

In [ ]:
# 13. Train
trainer = Trainer(config, pretrained=True)

print('Bắt đầu training...')
print(f'   Log lưu tại: {OUTPUT_DIR}/train.log')
print('─' * 60)

trainer.train()

In [ ]:
# 14. Evaluate
from vietocr.tool.predictor import Predictor
from tqdm import tqdm
import editdistance

config_eval = Cfg.load_config_from_name(MODEL_NAME)
config_eval['weights']            = os.path.join(OUTPUT_DIR, 'best_model.pth')
config_eval['device']             = 'cuda' if torch.cuda.is_available() else 'cpu'
config_eval['cnn']['pretrained']  = False

predictor = Predictor(config_eval)
print('Load model thành công!')


def evaluate(annotation_path, predictor, n_samples=None):
    samples = []
    with open(annotation_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                parts = line.split('\t', 1)
                if len(parts) == 2:
                    samples.append((parts[0].strip(), parts[1].strip()))

    if n_samples:
        samples = samples[:n_samples]

    correct, cer_total, total = 0, 0, len(samples)

    for img_path, true_label in tqdm(samples, desc='Evaluating'):
        try:
            img  = Image.open(img_path).convert('RGB')
            pred = predictor.predict(img)
            if pred.strip() == true_label.strip():
                correct += 1
            cer_total += editdistance.eval(true_label, pred) / max(len(true_label), 1)
        except:
            total -= 1

    accuracy = correct / total * 100 if total > 0 else 0
    avg_cer  = cer_total / total      if total > 0 else 1
    return accuracy, avg_cer, correct, total


print('\nĐang evaluate trên tập TEST (5000 mẫu)...')
acc, cer, correct, total = evaluate(TEST_ANNO, predictor, n_samples=5000)

print(f'\nKẾT QUẢ:')
print(f'   Accuracy : {acc:.2f}%  ({correct}/{total})')
print(f'   CER      : {cer:.4f}  (càng thấp càng tốt)')